In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

In [2]:
df = pd.read_excel('Online Retail.xlsx')

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.shape

(541909, 8)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [6]:
kolom_relevan = ['InvoiceNo', 'Description', 'Quantity']
df_selected = df[kolom_relevan].copy()
df_selected = df_selected.dropna(subset=['InvoiceNo', 'Description'])
df_selected['InvoiceNo'] = df_selected['InvoiceNo'].astype(str).str.strip()
df_selected['Description'] = df_selected['Description'].astype(str).str.strip()
df_clean = df_selected[~df_selected['InvoiceNo'].str.startswith('C')]
df_clean = df_clean[df_clean['Quantity'] > 0]

print(f'Sebelum clean: {df.shape}')
print(f'Sesudah clean: {df_clean.shape}')

Sebelum clean: (541909, 8)
Sesudah clean: (530693, 3)


In [7]:
basket = (df_clean.groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum()
          .unstack()
          .reset_index()
          .fillna(0)
          .set_index('InvoiceNo'))

df_onehot = basket > 0


In [8]:
frequent_itemsets = apriori(df_onehot, min_support=0.02, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5)

rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))

hasil_rules = (rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
               .sort_values('lift', ascending=False)
               .reset_index(drop=True))

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 250)
print(f'Jumlah aturan: {len(hasil_rules)}')
display(hasil_rules)

# Top 3 
top = hasil_rules.copy()
top['itemset'] = top.apply(lambda r: frozenset((r['antecedents'] + ', ' + r['consequents']).split(', ')), axis=1)
top['lift_round'] = top['lift'].round(6)
top3 = (top.sort_values(['lift_round', 'confidence'], ascending=False)
           .drop_duplicates('itemset').head(3)
           .drop(columns=['itemset', 'lift_round']).reset_index(drop=True))
top3.index += 1
display(top3)


Jumlah aturan: 61


,antecedents,consequents,support,confidence,lift
0,PINK REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER",0.026917,0.707572,18.551648
1,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER",PINK REGENCY TEACUP AND SAUCER,0.026917,0.705729,18.551648
2,GREEN REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER",0.026917,0.533990,17.950627
3,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER",GREEN REGENCY TEACUP AND SAUCER,0.026917,0.904841,17.950627
4,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.031436,0.623645,16.393893
...,...,...,...,...,...
56,JUMBO SHOPPER VINTAGE RED PAISLEY,JUMBO BAG RED RETROSPOT,0.033770,0.578723,5.570351
57,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER",REGENCY CAKESTAND 3 TIER,0.020262,0.531250,5.378205
58,JUMBO BAG APPLES,JUMBO BAG RED RETROSPOT,0.027116,0.558282,5.373600
59,RECYCLING BAG RETROSPOT,JUMBO BAG RED RETROSPOT,0.020610,0.554813,5.340206


,antecedents,consequents,support,confidence,lift
1,PINK REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER",0.026917,0.707572,18.551648
2,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.031436,0.826371,16.393893
3,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,0.027116,0.720317,15.886413
